In [ ]:
# !pip install youtube-transcript-api

In [3]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [5]:
## Document Loading

yt_api = YouTubeTranscriptApi()

vid_id = 'SfOaZIGJ_gs'

transcripts = yt_api.fetch(vid_id)

In [14]:
trns_txt = " ".join([doc.text for doc in transcripts])

In [15]:
## Text Splitting

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = splitter.split_text(trns_txt)

In [17]:
len(chunks)

88

In [18]:
embeddings = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')

c:\Users\ManishAgrahari\Desktop\my_learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1962.85it/s]


In [19]:
vector_store = FAISS.from_texts(chunks, embeddings)

In [21]:
retriever = vector_store.as_retriever(search_kwargs={'k':4})

In [22]:
retriever.invoke("how to get not afraid of AI")

[Document(id='34b4c987-269e-460c-b40e-ba3b7492ccb3', metadata={}, page_content="high school that it seemed to me that the obvious thing to do was learn to program. And I didn't know exactly what I was going to use that for, but there was this new frontier that seemed very high leverage and important to get good at. And right now, learning how to use AI tools is probably the most important specific hard skill to learn. And the difference between people who are really good at really AI native and think of everything in terms of those tools and don't is huge. There's other"),
 Document(id='2db60f02-d957-4c4b-92ca-fae5fed1163d', metadata={}, page_content="So, uh, that's something as well. But, you know, in terms of the specifics, like are you supposed to take the biology class or the physics class, I don't think it matters right now. >> And if I were to build on top of that, and when you say learn to adapt and change and learn AI tools faster, is there a path? I'm just looking for a light 

In [35]:
## Augmentation

prompt = PromptTemplate(
    template="""You are a helpful assistant. Answer using the context.
If the context is insufficient, say "I don't know."

Context: {context}
Question: {question}""",
    input_variables=["context", "question"],
)

In [36]:
ques = 'If how to start a company is there'

context = retriever.invoke(ques)

In [31]:
context = ' '.join(i.page_content for i in context)

In [37]:
final_prompt = prompt.invoke({'question': ques, 'context': context})

In [38]:
## Generation

llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0.7)

In [39]:
llm.invoke(final_prompt)

AIMessage(content='Starting a company—especially in a fast‑moving area like AI—looks a lot like what the speakers in the excerpts describe:\n\n1. **Pick a deep, valuable problem**  \n   Companies that “build on top of AI models and create huge value and very deep relationships with their customers” tend to succeed. Look for a use‑case where the AI can solve something that customers really care about and where you can become indispensable.\n\n2. **Take a long time horizon**  \n   One of the speakers said they spent *four and a half years* before releasing their first product. Be prepared for a long, uncertain development phase and resist the pressure to ship too early.\n\n3. **Think independently**  \n   Independent thinking helped them stay focused on the real problem rather than chasing hype. Question assumptions and be willing to go against the crowd if the data supports you.\n\n4. **Work super hard and be patient**  \n   Hard work combined with luck was highlighted as a key factor. 